In [0]:
customers = [
    (1, "Arjun", "Kochi"),
    (2, "Meera", "Chennai"),
    (3, "Rahul", "Bangalore"),
    (4, "Sneha", "Kochi"),
    (5, "Vikram", "Hyderabad"),
    (6, "Asha", "Chennai")
]
 
columns = ["customer_id", "customer_name", "city"]
 
df_customers = spark.createDataFrame(customers, columns)
df_customers.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|          1|        Arjun|    Kochi|
|          2|        Meera|  Chennai|
|          3|        Rahul|Bangalore|
|          4|        Sneha|    Kochi|
|          5|       Vikram|Hyderabad|
|          6|         Asha|  Chennai|
+-----------+-------------+---------+



In [0]:
orders = [
    (101, 1, 1001, 2),
    (102, 2, 1002, 1),
    (103, 1, 1003, 3),
    (104, 3, 1001, 1),
    (105, 5, 1004, 2),
    (106, 7, 1002, 1)
]
 
columns = ["order_id", "customer_id", "product_id", "quantity"]
 
df_orders = spark.createDataFrame(orders, columns)
df_orders.show()

+--------+-----------+----------+--------+
|order_id|customer_id|product_id|quantity|
+--------+-----------+----------+--------+
|     101|          1|      1001|       2|
|     102|          2|      1002|       1|
|     103|          1|      1003|       3|
|     104|          3|      1001|       1|
|     105|          5|      1004|       2|
|     106|          7|      1002|       1|
+--------+-----------+----------+--------+



In [0]:
products = [
    (1001, "Laptop", 50000),
    (1002, "Mobile", 20000),
    (1003, "Tablet", 30000),
    (1004, "Headphones", 5000)
]
 
columns = ["product_id", "product_name", "price"]
 
df_products = spark.createDataFrame(products, columns)
df_products.show()

+----------+------------+-----+
|product_id|product_name|price|
+----------+------------+-----+
|      1001|      Laptop|50000|
|      1002|      Mobile|20000|
|      1003|      Tablet|30000|
|      1004|  Headphones| 5000|
+----------+------------+-----+




# 🔹 Level 1: Basic Joins

-  Task 1
    
    Join orders + customers (inner join)

In [0]:
df_orders.join(df_customers,"customer_id","inner").show()

+-----------+--------+----------+--------+-------------+---------+
|customer_id|order_id|product_id|quantity|customer_name|     city|
+-----------+--------+----------+--------+-------------+---------+
|          1|     103|      1003|       3|        Arjun|    Kochi|
|          1|     101|      1001|       2|        Arjun|    Kochi|
|          2|     102|      1002|       1|        Meera|  Chennai|
|          3|     104|      1001|       1|        Rahul|Bangalore|
|          5|     105|      1004|       2|       Vikram|Hyderabad|
+-----------+--------+----------+--------+-------------+---------+



-  Task 2
 
    Join orders + customers (left join) Identify customers with no orders

In [0]:
new = df_customers.join(df_orders,"customer_id","left")
new.show()
new.filter(new.order_id.isNull()).show()

+-----------+-------------+---------+--------+----------+--------+
|customer_id|customer_name|     city|order_id|product_id|quantity|
+-----------+-------------+---------+--------+----------+--------+
|          1|        Arjun|    Kochi|     103|      1003|       3|
|          1|        Arjun|    Kochi|     101|      1001|       2|
|          2|        Meera|  Chennai|     102|      1002|       1|
|          3|        Rahul|Bangalore|     104|      1001|       1|
|          4|        Sneha|    Kochi|    NULL|      NULL|    NULL|
|          5|       Vikram|Hyderabad|     105|      1004|       2|
|          6|         Asha|  Chennai|    NULL|      NULL|    NULL|
+-----------+-------------+---------+--------+----------+--------+

+-----------+-------------+-------+--------+----------+--------+
|customer_id|customer_name|   city|order_id|product_id|quantity|
+-----------+-------------+-------+--------+----------+--------+
|          4|        Sneha|  Kochi|    NULL|      NULL|    NULL|
| 


# 🔹 Level 2: Multi-Table Join

-  Task 3

    Join: orders, customers, products
 
    Final columns: customer_name, product_name, quantity, price

In [0]:
df_customers.join(df_orders,"customer_id","outer").join(df_products,"product_id","outer").select("customer_name","product_name","quantity","price").show()

+-------------+------------+--------+-----+
|customer_name|product_name|quantity|price|
+-------------+------------+--------+-----+
|       Vikram|  Headphones|       2| 5000|
|        Arjun|      Laptop|       2|50000|
|        Rahul|      Laptop|       1|50000|
|        Arjun|      Tablet|       3|30000|
|        Meera|      Mobile|       1|20000|
|         NULL|      Mobile|       1|20000|
|        Sneha|        NULL|    NULL| NULL|
|         Asha|        NULL|    NULL| NULL|
+-------------+------------+--------+-----+



# 🔹 Level 3: Calculations

- Task 4
    
    Create column: total_amount = quantity * price

In [0]:
df = df_orders.join(df_products,"product_id","inner")
df.withColumn("total_amount",df.quantity*df.price).show()


+----------+--------+-----------+--------+------------+-----+------------+
|product_id|order_id|customer_id|quantity|product_name|price|total_amount|
+----------+--------+-----------+--------+------------+-----+------------+
|      1001|     101|          1|       2|      Laptop|50000|      100000|
|      1002|     102|          2|       1|      Mobile|20000|       20000|
|      1003|     103|          1|       3|      Tablet|30000|       90000|
|      1001|     104|          3|       1|      Laptop|50000|       50000|
|      1004|     105|          5|       2|  Headphones| 5000|       10000|
|      1002|     106|          7|       1|      Mobile|20000|       20000|
+----------+--------+-----------+--------+------------+-----+------------+



# 🔹 Level 4: Analytics
- Task 5
    
    Total revenue per customer


In [0]:
from pyspark.sql.functions import sum
df = df_customers.join(df_orders,"customer_id","inner").join(df_products,"product_id","inner")
df.groupBy("customer_id").agg(sum("price").alias("total_revenue")).show()

+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|          1|        80000|
|          2|        20000|
|          3|        50000|
|          5|         5000|
+-----------+-------------+



- Task 6
    
    Total revenue per city


In [0]:
from pyspark.sql.functions import sum
df = df_customers.join(df_orders,"customer_id","inner").join(df_products,"product_id","inner")
df.groupBy("city").agg(sum("price").alias("total_revenue")).show()


+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|    Kochi|        80000|
|  Chennai|        20000|
|Bangalore|        50000|
|Hyderabad|         5000|
+---------+-------------+



- Task 7
    
    Total revenue per product

In [0]:
from pyspark.sql.functions import sum
df = df_customers.join(df_orders,"customer_id","inner").join(df_products,"product_id","inner")
df.groupBy("product_id").agg(sum("price").alias("total_revenue")).show()

+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|      1001|       100000|
|      1002|        20000|
|      1003|        30000|
|      1004|         5000|
+----------+-------------+

